# Grazioso Salvare Rescue Dashboard
Created by Zander Taylor. Run the code cells in order. This private course copy contains database credentials; remove the password before publishing.


In [ ]:
# Imports and configuration. Run this notebook from the extracted project folder.
import base64
import json
import logging
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import dash_leaflet as dl
from dash import Dash, dcc, html, dash_table, Input, Output, jupyter_dash
from pymongo.errors import PyMongoError
from CRUD_Python_Module import AnimalShelter
from rescue_queries import rescue_query

# Required course account instantiated explicitly. Do not publish real credentials.
username = "aacuser"
password = "AACdb!340_ZT26"
CREATOR_NAME = "Zander Taylor"
db = AnimalShelter(username, password, auth_source="aac")
# If your account was created in admin, use auth_source="admin" instead.
FIELDS = ["animal_id", "name", "animal_type", "breed", "sex_upon_outcome",
          "age_upon_outcome_in_weeks", "outcome_type", "location_lat", "location_long"]


def load_records(rescue_type):
    records = db.read(rescue_query(rescue_type), {"_id": 0})
    frame = pd.DataFrame.from_records(records).drop(columns=[""], errors="ignore")
    frame = frame.reindex(columns=FIELDS)
    for column in ["age_upon_outcome_in_weeks", "location_lat", "location_long"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    # JSON converts missing values to null so Dash receives valid browser data.
    return json.loads(frame.to_json(orient="records"))

try:
    initial_records = load_records("reset")
    initial_status = f"{len(initial_records):,} records loaded"
except PyMongoError:
    logging.exception("Initial database query failed")
    initial_records = []
    initial_status = "Database unavailable. Check credentials, MongoDB, and authentication database."


In [ ]:
# View: branding, rescue controls, interactive table, and linked charts.
app = Dash(__name__)
logo = base64.b64encode(Path("Grazioso Salvare Logo.png").read_bytes()).decode()
app.layout = html.Div([
    html.Header([
        html.A(html.Img(src=f"data:image/png;base64,{logo}", style={"height": "105px"}),
               href="https://www.snhu.edu", target="_blank"),
        html.Div([html.H1("Grazioso Salvare Rescue Dashboard"),
                  html.P(f"Created by {CREATOR_NAME} | CS 340 Project Two")])
    ], style={"display": "flex", "alignItems": "center", "gap": "24px"}),
    html.P("Choose a rescue profile, then sort or filter the table to explore candidates."),
    dcc.RadioItems(id="rescue-filter", options=[
        {"label": "Reset / All animals", "value": "reset"},
        {"label": "Water Rescue", "value": "water"},
        {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
        {"label": "Disaster or Individual Tracking", "value": "disaster"}
    ], value="reset", inline=True, inputStyle={"marginLeft": "18px", "marginRight": "6px"}),
    html.P(id="status", children=initial_status, role="status"),
    dcc.Loading(dash_table.DataTable(
        id="datatable-id", columns=[{"name": c.replace("_", " ").title(), "id": c,
           "type": "numeric" if c in ["age_upon_outcome_in_weeks", "location_lat", "location_long"] else "text"}
           for c in FIELDS], data=initial_records,
        page_size=10, page_action="native", sort_action="native", sort_mode="multi",
        filter_action="native", filter_query="", sort_by=[], page_current=0,
        style_table={"overflowX": "auto"},
        style_cell={"fontFamily": "Arial", "fontSize": 12, "padding": "8px", "textAlign": "left",
                    "minWidth": "100px", "maxWidth": "250px", "whiteSpace": "normal"},
        style_header={"backgroundColor": "#edf0f3", "fontWeight": "bold"},
        style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f8f9fa"}]
    )),
    html.Div([
        html.Div([dcc.Graph(id="graph-id")], style={"flex": "1", "minWidth": "360px"}),
        html.Div([html.H3("Animal locations"), html.P(id="map-status"),
            dl.Map([dl.TileLayer(), dl.LayerGroup(id="map-markers")],
                   id="animal-map", center=[30.2672, -97.7431], zoom=9,
                   style={"height": "350px", "width": "100%"})],
                 style={"flex": "1", "minWidth": "360px"})
    ], style={"display": "flex", "flexWrap": "wrap", "gap": "24px"})
], style={"fontFamily": "Arial", "maxWidth": "1450px", "margin": "auto", "padding": "24px"})


In [ ]:
# Controller: query through the Project One module, then refresh the table.
@app.callback(
    Output("datatable-id", "data"), Output("datatable-id", "page_current"),
    Output("datatable-id", "filter_query"), Output("datatable-id", "sort_by"),
    Output("status", "children"), Input("rescue-filter", "value"),
    prevent_initial_call=True)
def update_table(rescue_type):
    try:
        rows = load_records(rescue_type)
        return rows, 0, "", [], f"{len(rows):,} records match this rescue profile"
    except PyMongoError:
        logging.exception("Rescue filter query failed")
        return [], 0, "", [], "Database query failed. Check the connection and credentials."


# derived_virtual_data includes all matching rows across pages, after table filtering.
@app.callback(Output("graph-id", "figure"), Output("map-markers", "children"),
              Output("map-status", "children"),
              Input("datatable-id", "derived_virtual_data"), Input("datatable-id", "data"))
def update_charts(virtual_rows, table_rows):
    rows = table_rows if virtual_rows is None else virtual_rows
    frame = pd.DataFrame(rows or [], columns=FIELDS)
    if frame.empty:
        figure = go.Figure()
        figure.add_annotation(text="No matching animals", showarrow=False)
        return figure, [], "No matching animals to map."
    counts = frame["breed"].fillna("Unknown").value_counts().rename_axis("Breed").reset_index(name="Animals")
    # Keep long unfiltered breed distributions readable while preserving totals.
    if len(counts) > 10:
        counts = pd.concat([counts.head(10), pd.DataFrame([{"Breed": "Other breeds", "Animals": int(counts.iloc[10:]["Animals"].sum())}])], ignore_index=True)
    figure = px.pie(counts, names="Breed", values="Animals", hole=0.35,
                    title=f"Breed distribution ({len(rows):,} animals)")
    figure.update_layout(margin=dict(l=12, r=12, t=55, b=12), height=430)
    lat = pd.to_numeric(frame["location_lat"], errors="coerce")
    lon = pd.to_numeric(frame["location_long"], errors="coerce")
    valid = frame.loc[lat.between(-90, 90) & lon.between(-180, 180)].copy()
    valid["location_lat"] = lat.loc[valid.index]
    valid["location_long"] = lon.loc[valid.index]
    # GeoJSON clusters keep the initial 10,000-record map usable.
    features = [{"type": "Feature", "geometry": {"type": "Point", "coordinates": [r["location_long"], r["location_lat"]]},
                 "properties": {"animal_id": str(r["animal_id"] or ""), "breed": str(r["breed"] or "")}}
                for r in valid.to_dict("records")]
    layer = dl.GeoJSON(data={"type": "FeatureCollection", "features": features},
                       cluster=True, zoomToBounds=bool(features), superClusterOptions={"radius": 50})
    return figure, [layer], f"{len(valid):,} locations mapped; {len(frame)-len(valid):,} records lack valid coordinates."


In [ ]:
# Run last. Modern Dash includes Jupyter support without JupyterDash.
# For Codio, open port 8050 using its application preview when needed.
# If automatic proxy discovery is needed, first run: jupyter_dash.infer_jupyter_proxy_config()
app.run(host="0.0.0.0", port=8050, debug=False, jupyter_mode="external")
